# Phase 1: Baseline scorecard

Compares `dispute_triage@v1` and `@v2`, each with default and minimal reasoning effort, on the synthetic disputes in `data/samples/disputes_v1.jsonl`.

All logic lives in `src/atlas/`. Set `ATLAS_MOCK=1` to run offline (CI does this).

In [ ]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from atlas.scorecard import run_experiment, summarize  # noqa: E402  (loads .env)

In [ ]:
cases = [json.loads(line) for line in open(ROOT / "data/samples/disputes_v1.jsonl")]
model = os.environ["ATLAS_MODELS"]
mode = "mock" if os.environ.get("ATLAS_MOCK") == "1" else "live"
print(f"{len(cases)} cases, mode={mode}")

In [ ]:
variants = [
    {"prompt_id": prompt_id, "model": model, "params": params}
    for prompt_id in ["dispute_triage@v1", "dispute_triage@v2"]
    for params in [{}, {"reasoning_effort": "minimal"}]
]
df = run_experiment(variants, cases)
summary = summarize(df)
summary.round(3)

## Misses

Every case a variant got wrong, including invalid JSON and provider errors.

In [ ]:
misses = df[~df["correct"]][["variant", "case_id", "label", "predicted", "valid_json", "error"]]
misses

In [ ]:
out = ROOT / "results"
out.mkdir(exist_ok=True)
df.to_csv(out / f"phase01_cases_{mode}.csv", index=False)
summary.to_csv(out / f"phase01_summary_{mode}.csv")
print("saved to", out)